#### Imports

##### Core Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import warnings

warnings.filterwarnings("ignore")

##### TensorFlow and Keras imports

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16, DenseNet201, ResNet50, EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    GlobalAveragePooling2D,
    BatchNormalization,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

##### Scikit-learn metrics

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

##### Explainability libraries

In [ ]:
import shap
from lime import lime_image
from skimage.segmentation import mark_boundaries

##### Random Seeds and Plot Config

In [ ]:
# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Configure plotting style
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
sns.set_context("notebook", font_scale=1.2)

#### Data Loading and Preprocessing

In [ ]:
# Split training data: train/validation (80/20)


def load_data():
    base_dir = "dataset"
    train_dir = os.path.join(base_dir, "Training")
    test_dir = os.path.join(base_dir, "Testing")

    # Data augmentation for training (helps with generalization)
    train_datagen = ImageDataGenerator(
        rescale=1.0 / 255,  # Normalize pixel values to [0,1]
        rotation_range=20,  # Random rotation up to 20 degrees
        width_shift_range=0.2,  # Random horizontal shift
        height_shift_range=0.2,  # Random vertical shift
        shear_range=0.2,  # Random shear transformation
        zoom_range=0.2,  #  Random zoom
        horizontal_flip=True,  # Random horizontal flip
        validation_split=0.2,  # Reserve 20% for validation
    )

    # Only rescaling for test data (no augmentation)
    test_datagen = ImageDataGenerator(rescale=1.0 / 255)

    # Create generators
    train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=(224, 224),  # Resize all images to 224x224
        batch_size=32,
        class_mode="binary",  # Binary classification (Normal vs Stroke)
        subset="training",  # Training subset
        seed=42,
    )

    validation_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=(224, 224),
        batch_size=32,
        class_mode="binary",
        subset="validation",  # Validation subset
        seed=42,
    )

    test_generator = test_datagen.flow_from_directory(
        test_dir,
        target_size=(224, 224),
        batch_size=32,
        class_mode="binary",
        shuffle=False,  # Don't shuffle for evaluation
        seed=42,
    )

    return train_generator, validation_generator, test_generator


# Load the data
train_generator, validation_generator, test_generator = load_data()

#### Dataset Statistics

In [ ]:
def plot_dataset_stats(train_gen, valid_gen, test_gen):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    datasets = ["Training", "Validation", "Testing"]
    samples = [train_gen.samples, valid_gen.samples, test_gen.samples]
    class_names = list(train_gen.class_indices.keys())
    colors = ["#2E86AB", "#A23B72", "#F18F01"]

    # Plot 1-3: Dataset sizes
    for i, (ax, name, count) in enumerate(zip(axes, datasets, samples)):
        ax.bar(name, count, color=colors[i], edgecolor="black", linewidth=1.5)
        ax.set_ylabel("Number of Images", fontsize=11)
        ax.set_title(f"{name} Set\n{count} samples", fontsize=12, fontweight="bold")
        ax.grid(axis="y", alpha=0.3)

    fig.suptitle("Dataset Distribution", fontsize=14, fontweight="bold", y=1.05)
    plt.tight_layout()
    plt.show()

    # Plot 4: Class distribution
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))

    class_counts = []
    for class_name in class_names:
        train_count = len(
            [
                f
                for f in os.listdir(os.path.join("dataset/Training", class_name))
                if f.endswith((".png", ".jpg", ".jpeg"))
            ]
        )
        test_count = len(
            [
                f
                for f in os.listdir(os.path.join("dataset/Testing", class_name))
                if f.endswith((".png", ".jpg", ".jpeg"))
            ]
        )
        class_counts.append([train_count, test_count])

    x = np.arange(len(class_names))
    width = 0.35

    ax.bar(
        x - width / 2,
        [c[0] for c in class_counts],
        width,
        label="Training",
        color="#2E86AB",
        edgecolor="black",
        linewidth=1,
    )
    ax.bar(
        x + width / 2,
        [c[1] for c in class_counts],
        width,
        label="Testing",
        color="#F18F01",
        edgecolor="black",
        linewidth=1,
    )

    ax.set_xlabel("Class", fontsize=12)
    ax.set_ylabel("Number of Images", fontsize=12)
    ax.set_title("Class Distribution Across Datasets", fontsize=14, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(class_names)
    ax.legend(loc="upper right", frameon=True, fancybox=True, shadow=True)
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()


# Display dataset statistics
plot_dataset_stats(train_generator, validation_generator, test_generator)

#### Model Architecture

In [ ]:
def build_model(base_model_name, input_shape=(224, 224, 3)):

    # Dictionary mapping model names to their classes
    base_models = {
        "VGG16": VGG16,
        "DenseNet201": DenseNet201,
        "ResNet50": ResNet50,
        "EfficientNetB0": EfficientNetB0,
    }

    # Load pre-trained base model (without top classification layer)
    base_model = base_models[base_model_name](
        weights="imagenet", include_top=False, input_shape=input_shape
    )
    base_model.trainable = False  # Freeze base model weights initially

    # Build custom classification head
    inputs = tf.keras.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)  # Reduce spatial dimensions
    x = BatchNormalization()(x)  # Normalize activations
    x = Dropout(0.5)(x)  # Prevent overfitting
    x = Dense(
        128, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(0.01)
    )(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    outputs = Dense(1, activation="sigmoid")(x)  # Binary output

    model = Model(inputs, outputs)

    # Compile model with Adam optimizer and binary cross-entropy loss
    model.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.Precision(),
            tf.keras.metrics.Recall(),
            tf.keras.metrics.AUC(),
        ],
    )

    return model

#### MODEL TRAINING

In [ ]:
def train_model(model_name, train_gen, valid_gen):
    print(f"\n{'=' * 60}")
    print(f" Training {model_name}")
    print(f"{'=' * 60}\n")

    model = build_model(model_name)

    # Early stopping: stop if validation accuracy doesn't improve for 5 epochs
    early_stopping = EarlyStopping(
        monitor="val_accuracy", patience=5, restore_best_weights=True, verbose=0
    )

    # Reduce learning rate when validation loss plateaus
    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss", factor=0.2, patience=3, min_lr=1e-7, verbose=0
    )

    # Train the model
    history = model.fit(
        train_gen,
        epochs=2,  # Reduced for demonstration; increase for full training
        validation_data=valid_gen,
        callbacks=[early_stopping, reduce_lr],
        verbose=1,  # Show progress
    )

    return model, history

##### VGG16

In [ ]:
model_vgg16, history_vgg16 = train_model("VGG16", train_generator, validation_generator)

##### DENSENET201

In [ ]:
model_densenet, history_densenet = train_model(
    "DenseNet201", train_generator, validation_generator
)

##### RESNET50

In [ ]:
model_resnet, history_resnet = train_model(
    "ResNet50", train_generator, validation_generator
)

##### EFFICIENTNETB0

In [ ]:
model_efficientnet, history_efficientnet = train_model(
    "EfficientNetB0", train_generator, validation_generator
)

#### Training History (Single Model ~VGG16)

In [ ]:
def plot_training_history(history, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    epochs = range(1, len(history.history["accuracy"]) + 1)

    # Accuracy plot
    axes[0].plot(
        epochs,
        history.history["accuracy"],
        "o-",
        color="#2E86AB",
        linewidth=2,
        markersize=4,
        label="Training",
    )
    axes[0].plot(
        epochs,
        history.history["val_accuracy"],
        "o-",
        color="#F18F01",
        linewidth=2,
        markersize=4,
        label="Validation",
    )
    axes[0].set_xlabel("Epochs", fontsize=11)
    axes[0].set_ylabel("Accuracy", fontsize=11)
    axes[0].set_title(f"{model_name} - Accuracy", fontsize=12, fontweight="bold")
    axes[0].legend(loc="lower right", frameon=True)
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim([0, 1])

    # Loss plot
    axes[1].plot(
        epochs,
        history.history["loss"],
        "o-",
        color="#2E86AB",
        linewidth=2,
        markersize=4,
        label="Training",
    )
    axes[1].plot(
        epochs,
        history.history["val_loss"],
        "o-",
        color="#F18F01",
        linewidth=2,
        markersize=4,
        label="Validation",
    )
    axes[1].set_xlabel("Epochs", fontsize=11)
    axes[1].set_ylabel("Loss", fontsize=11)
    axes[1].set_title(f"{model_name} - Loss", fontsize=12, fontweight="bold")
    axes[1].legend(loc="upper right", frameon=True)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


# Display training history for VGG16
plot_training_history(history_vgg16, "VGG16")

#### Model Evaluation Function

In [ ]:
def evaluate_model(model, test_gen, model_name):
    y_pred_prob = model.predict(test_gen, verbose=0)
    y_pred = (y_pred_prob > 0.5).astype(int).flatten()
    y_true = test_gen.classes

    # Calculate all metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_pred_prob)

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "ROC-AUC": roc_auc,
    }

    return metrics, y_true, y_pred, y_pred_prob

#### Model Evaluation on all Models

In [ ]:
metrics_vgg16, y_true, y_pred_vgg16, y_prob_vgg16 = evaluate_model(
    model_vgg16, test_generator, "VGG16"
)

metrics_densenet, _, y_pred_densenet, y_prob_densenet = evaluate_model(
    model_densenet, test_generator, "DenseNet201"
)

metrics_resnet, _, y_pred_resnet, y_prob_resnet = evaluate_model(
    model_resnet, test_generator, "ResNet50"
)

metrics_efficientnet, _, y_pred_efficientnet, y_prob_efficientnet = evaluate_model(
    model_efficientnet, test_generator, "EfficientNetB0"
)

#### Confusion Matrix

In [ ]:
def plot_confusion_matrix(y_true, y_pred, model_name):
    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(6, 5))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=True,
        xticklabels=["Normal", "Stroke"],
        yticklabels=["Normal", "Stroke"],
        square=True,
        linewidths=1,
        linecolor="black",
        annot_kws={"size": 14, "weight": "bold"},
    )

    ax.set_xlabel("Predicted Label", fontsize=12)
    ax.set_ylabel("True Label", fontsize=12)
    ax.set_title(f"{model_name} - Confusion Matrix", fontsize=14, fontweight="bold")

    plt.tight_layout()
    plt.show()


# Display confusion matrices
plot_confusion_matrix(y_true, y_pred_vgg16, "VGG16")
plot_confusion_matrix(y_true, y_pred_densenet, "DenseNet201")
plot_confusion_matrix(y_true, y_pred_resnet, "ResNet50")
plot_confusion_matrix(y_true, y_pred_efficientnet, "EfficientNetB0")

#### ROC Curve

In [ ]:
def plot_roc_curve(y_true, y_prob, model_name):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)

    fig, ax = plt.subplots(figsize=(7, 6))

    ax.plot(fpr, tpr, color="#2E86AB", lw=3, label=f"ROC Curve (AUC = {roc_auc:.3f})")
    ax.plot(
        [0, 1], [0, 1], color="#A23B72", lw=2, linestyle="--", label="Random Classifier"
    )
    ax.fill_between(fpr, tpr, alpha=0.2, color="#2E86AB")

    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate", fontsize=12)
    ax.set_title(f"{model_name} - ROC Curve", fontsize=14, fontweight="bold")
    ax.legend(loc="lower right", frameon=True, fancybox=True, shadow=True)
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


# Display ROC curves
plot_roc_curve(y_true, y_prob_vgg16, "VGG16")
plot_roc_curve(y_true, y_prob_densenet, "DenseNet201")
plot_roc_curve(y_true, y_prob_resnet, "ResNet50")
plot_roc_curve(y_true, y_prob_efficientnet, "EfficientNetB0")

#### Precision-Recall Curve

In [ ]:
def plot_pr_curve(y_true, y_prob, model_name):
    precision, recall, _ = precision_recall_curve(y_true, y_prob)

    fig, ax = plt.subplots(figsize=(7, 6))

    ax.plot(recall, precision, color="#F18F01", lw=3, label="Precision-Recall Curve")
    ax.set_xlabel("Recall", fontsize=12)
    ax.set_ylabel("Precision", fontsize=12)
    ax.set_title(
        f"{model_name} - Precision-Recall Curve", fontsize=14, fontweight="bold"
    )
    ax.legend(loc="lower left", frameon=True, fancybox=True, shadow=True)
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


# Display Precision-Recall curves
plot_pr_curve(y_true, y_prob_vgg16, "VGG16")
plot_pr_curve(y_true, y_prob_densenet, "DenseNet201")
plot_pr_curve(y_true, y_prob_resnet, "ResNet50")
plot_pr_curve(y_true, y_prob_efficientnet, "EfficientNetB0")

#### Grad-CAM heatmap - Generic version

In [ ]:
def make_gradcam_heatmap(img_array, model, pred_index=None):

    # Find the last convolutional layer
    last_conv_layer = None
    base_model_names = ["vgg16", "resnet50", "densenet201", "efficientnetb0"]

    for base_name in base_model_names:
        if base_name in [l.name for l in model.layers]:
            base_model = model.get_layer(base_name)
            # Find last conv layer in base model
            for layer in reversed(base_model.layers):
                if isinstance(layer, tf.keras.layers.Conv2D):
                    last_conv_layer = f"{base_name}/{layer.name}"
                    break
            break

    # Fallback: look in main model
    if last_conv_layer is None:
        for layer in reversed(model.layers):
            if isinstance(layer, tf.keras.layers.Conv2D):
                last_conv_layer = layer.name
                break

    if last_conv_layer is None:
        print(f"Could not find convolutional layer in {model.name}")
        return None

    # Create gradient model
    grad_model = tf.keras.models.Model(
        inputs=[model.inputs[0]],
        outputs=[model.get_layer(last_conv_layer).output, model.output],
    )

    img_array = tf.cast(img_array, tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(img_array)
        conv_outputs, predictions = grad_model(img_array, training=False)

        if predictions.shape[-1] > 1:
            pred_index = pred_index or tf.argmax(predictions[0])
            class_channel = predictions[:, pred_index]
        else:
            class_channel = predictions[:, 0]

    grads = tape.gradient(class_channel, conv_outputs)

    if grads is None:
        print("Gradient computation failed")
        return None

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)

    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + tf.keras.backend.epsilon())

    return heatmap.numpy()


# Display Grad-CAM visualization
def display_gradcam(model, model_name, test_gen):
    test_gen.reset()

    # Get a sample image
    img_array, label = next(test_gen)
    img = img_array[0].copy()

    # Prepare display image
    if img.max() <= 1.0:
        img_display = img.copy()
    else:
        img_display = img / 255.0

    # Create heatmap
    heatmap = make_gradcam_heatmap(np.expand_dims(img, axis=0), model)

    if heatmap is None:
        print(f"Grad-CAM failed for {model_name}")
        return

    # Resize and apply colormap
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    # Superimpose heatmap on original image
    if img.max() <= 1.0:
        superimposed_img = heatmap * 0.4 + img_display * 255
    else:
        superimposed_img = heatmap * 0.4 + img_display

    superimposed_img = np.clip(superimposed_img, 0, 255).astype("uint8")

    # Display results
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(img_display)
    axes[0].set_title(f"{model_name} - Original", fontsize=12, fontweight="bold")
    axes[0].axis("off")

    axes[1].imshow(heatmap)
    axes[1].set_title("Grad-CAM Heatmap", fontsize=12, fontweight="bold")
    axes[1].axis("off")

    axes[2].imshow(superimposed_img)
    axes[2].set_title("Overlay", fontsize=12, fontweight="bold")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()


# Specialized Grad-CAM for VGG16 (more reliable)
def display_gradcam_vgg16(model, model_name, test_gen):
    test_gen.reset()

    img_array, label = next(test_gen)
    img = img_array[0].copy()

    if img.max() <= 1.0:
        img_display = img.copy()
    else:
        img_display = img / 255.0

    # Access VGG16 base model
    vgg_base = model.get_layer("vgg16")

    # Create gradient model targeting block5_conv3
    grad_model = tf.keras.models.Model(
        inputs=[vgg_base.input],
        outputs=[vgg_base.get_layer("block5_conv3").output, model.output],
    )

    img_input = tf.cast(np.expand_dims(img, axis=0), tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(img_input)
        conv_outputs, predictions = grad_model(img_input, training=False)
        class_channel = predictions[:, 0]

    grads = tape.gradient(class_channel, conv_outputs)

    if grads is None:
        print("Gradient computation failed for VGG16")
        return

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + tf.keras.backend.epsilon())
    heatmap = heatmap.numpy()

    # Resize and apply colormap
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    # Superimpose
    if img.max() <= 1.0:
        superimposed_img = heatmap * 0.4 + img_display * 255
    else:
        superimposed_img = heatmap * 0.4 + img_display

    superimposed_img = np.clip(superimposed_img, 0, 255).astype("uint8")

    # Display
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(img_display)
    axes[0].set_title(f"{model_name} - Original", fontsize=12, fontweight="bold")
    axes[0].axis("off")

    axes[1].imshow(heatmap)
    axes[1].set_title("Grad-CAM Heatmap", fontsize=12, fontweight="bold")
    axes[1].axis("off")

    axes[2].imshow(superimposed_img)
    axes[2].set_title("Overlay", fontsize=12, fontweight="bold")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()


# Run Grad-CAM visualizations
display_gradcam_vgg16(model_vgg16, "VGG16", test_generator)
display_gradcam(model_densenet, "DenseNet201", test_generator)
display_gradcam(model_resnet, "ResNet50", test_generator)
display_gradcam(model_efficientnet, "EfficientNetB0", test_generator)

#### LIME Explanations

In [ ]:
def display_lime(model, model_name, test_gen):
    test_gen.reset()
    img_array, label = next(test_gen)
    img = img_array[0]

    def predict_fn(images):
        return model.predict(images, verbose=0)

    explainer = lime_image.LimeImageExplainer()

    explanation = explainer.explain_instance(
        img, predict_fn, top_labels=1, hide_color=0, num_samples=500, random_seed=42
    )

    # Positive only
    temp, mask = explanation.get_image_and_mask(
        explanation.top_labels[0], positive_only=True, num_features=5, hide_rest=False
    )

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(img)
    axes[0].set_title(f"{model_name} - Original Image", fontsize=12, fontweight="bold")
    axes[0].axis("off")

    axes[1].imshow(mark_boundaries(temp / 2 + 0.5, mask))
    axes[1].set_title("LIME Explanation", fontsize=12, fontweight="bold")
    axes[1].axis("off")

    # Positive and negative
    temp, mask = explanation.get_image_and_mask(
        explanation.top_labels[0], positive_only=False, num_features=5, hide_rest=False
    )

    axes[2].imshow(mark_boundaries(temp / 2 + 0.5, mask))
    axes[2].set_title("LIME (Positive + Negative)", fontsize=12, fontweight="bold")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()


# Run LIME explanations
display_lime(model_vgg16, "VGG16", test_generator)
display_lime(model_densenet, "DenseNet201", test_generator)
display_lime(model_resnet, "ResNet50", test_generator)
display_lime(model_efficientnet, "EfficientNetB0", test_generator)

#### SHAP Explaination

In [ ]:
def display_shap(model, model_name, test_gen):
    test_gen.reset()
    background, _ = next(test_gen)
    img_array, label = next(test_gen)
    img = img_array[0]

    def predict_fn(images):
        return model.predict(images, verbose=0)

    masker = shap.maskers.Image("inpaint_telea", img_array[0].shape)
    explainer = shap.Explainer(predict_fn, masker, output_names=["Normal", "Stroke"])

    shap_values = explainer(
        img_array[:1],
        max_evals=500,
        batch_size=50,
        outputs=shap.Explanation.argsort.flip[:1],
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].imshow(img)
    axes[0].set_title(f"{model_name} - Original Image", fontsize=12, fontweight="bold")
    axes[0].axis("off")

    shap.image_plot(shap_values, img_array[:1], show=False)

    plt.tight_layout()
    plt.show()


# Run SHAP explanations
display_shap(model_vgg16, "VGG16", test_generator)
display_shap(model_densenet, "DenseNet201", test_generator)
display_shap(model_resnet, "ResNet50", test_generator)
display_shap(model_efficientnet, "EfficientNetB0", test_generator)

#### Results

In [ ]:
def display_results_table(metrics_list):
    df = pd.DataFrame(metrics_list)
    df = df.round(4)

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.axis("tight")
    ax.axis("off")

    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        cellLoc="center",
        loc="center",
        colWidths=[0.15] * len(df.columns),
    )

    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.5)

    # Style header row
    for i, key in enumerate(df.columns):
        table[(0, i)].set_facecolor("#2E86AB")
        table[(0, i)].set_text_props(weight="bold", color="white")

    # Style alternating rows
    for i in range(1, len(df) + 1):
        for j in range(len(df.columns)):
            table[(i, j)].set_facecolor("#f8f9fa" if i % 2 == 0 else "#e9ecef")

    ax.set_title("Model Performance Comparison", fontsize=14, fontweight="bold", pad=20)

    plt.tight_layout()
    plt.show()


all_metrics = [metrics_vgg16, metrics_densenet, metrics_resnet, metrics_efficientnet]
display_results_table(all_metrics)

#### Model Comparison

In [ ]:
def plot_model_comparison(metrics_list):
    df = pd.DataFrame(metrics_list)
    df.set_index("Model", inplace=True)

    metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]

    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()

    colors = ["#2E86AB", "#A23B72", "#F18F01", "#6C9A8F"]

    for i, metric in enumerate(metrics_to_plot):
        ax = axes[i]
        bars = ax.bar(
            df.index, df[metric], color=colors, edgecolor="black", linewidth=1.5
        )
        ax.set_ylabel("Score", fontsize=11)
        ax.set_title(metric, fontsize=13, fontweight="bold")
        ax.set_ylim([0, 1])
        ax.grid(axis="y", alpha=0.3)

        # Add value labels on bars
        for bar, val in zip(bars, df[metric]):
            height = bar.get_height()
            ax.text(
                bar.get_x() + bar.get_width() / 2.0,
                height + 0.02,
                f"{val:.3f}",
                ha="center",
                va="bottom",
                fontsize=10,
                fontweight="bold",
            )

    axes[-1].axis("off")  # Hide unused subplot

    fig.suptitle(
        "Model Performance Comparison Across All Metrics",
        fontsize=16,
        fontweight="bold",
        y=1.02,
    )
    plt.tight_layout()
    plt.show()


plot_model_comparison(all_metrics)

#### Training History

In [ ]:
def plot_training_comparison(histories_dict):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    colors = ["#2E86AB", "#A23B72", "#F18F01", "#6C9A8F"]

    # Validation accuracy comparison
    for i, (model_name, history) in enumerate(histories_dict.items()):
        epochs = range(1, len(history.history["accuracy"]) + 1)
        axes[0].plot(
            epochs,
            history.history["val_accuracy"],
            "o-",
            color=colors[i],
            linewidth=2,
            markersize=4,
            label=model_name,
        )

    axes[0].set_xlabel("Epochs", fontsize=12)
    axes[0].set_ylabel("Validation Accuracy", fontsize=12)
    axes[0].set_title("Validation Accuracy Comparison", fontsize=14, fontweight="bold")
    axes[0].legend(loc="lower right", frameon=True, fancybox=True, shadow=True)
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim([0, 1])

    # Validation loss comparison
    for i, (model_name, history) in enumerate(histories_dict.items()):
        epochs = range(1, len(history.history["loss"]) + 1)
        axes[1].plot(
            epochs,
            history.history["val_loss"],
            "o-",
            color=colors[i],
            linewidth=2,
            markersize=4,
            label=model_name,
        )

    axes[1].set_xlabel("Epochs", fontsize=12)
    axes[1].set_ylabel("Validation Loss", fontsize=12)
    axes[1].set_title("Validation Loss Comparison", fontsize=14, fontweight="bold")
    axes[1].legend(loc="upper right", frameon=True, fancybox=True, shadow=True)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


histories = {
    "VGG16": history_vgg16,
    "DenseNet201": history_densenet,
    "ResNet50": history_resnet,
    "EfficientNetB0": history_efficientnet,
}

plot_training_comparison(histories)

#### ROC Curve Comparison

In [ ]:
def plot_roc_comparison(y_true, y_probs_dict):
    fig, ax = plt.subplots(figsize=(10, 8))

    colors = ["#2E86AB", "#A23B72", "#F18F01", "#6C9A8F"]

    for i, (model_name, y_prob) in enumerate(y_probs_dict.items()):
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        roc_auc = auc(fpr, tpr)
        ax.plot(
            fpr,
            tpr,
            color=colors[i],
            lw=2.5,
            label=f"{model_name} (AUC = {roc_auc:.3f})",
        )

    ax.plot(
        [0, 1], [0, 1], color="#A23B72", lw=2, linestyle="--", label="Random Classifier"
    )

    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate", fontsize=12)
    ax.set_title("ROC Curves - All Models Comparison", fontsize=14, fontweight="bold")
    ax.legend(loc="lower right", frameon=True, fancybox=True, shadow=True)
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


y_probs = {
    "VGG16": y_prob_vgg16,
    "DenseNet201": y_prob_densenet,
    "ResNet50": y_prob_resnet,
    "EfficientNetB0": y_prob_efficientnet,
}

plot_roc_comparison(y_true, y_probs)

#### Best Model Selection

In [ ]:
def select_best_model(metrics_list):
    df = pd.DataFrame(metrics_list)
    df.set_index("Model", inplace=True)

    # Calculate average score across main metrics
    df["Avg_Score"] = df[["Accuracy", "Precision", "Recall", "F1-Score"]].mean(axis=1)
    best_model = df["Avg_Score"].idxmax()
    best_score = df.loc[best_model, "Avg_Score"]

    # Create visualization
    fig, ax = plt.subplots(figsize=(10, 6))

    bars = ax.bar(
        df.index,
        df["Avg_Score"],
        color=["#2E86AB" if m != best_model else "#F18F01" for m in df.index],
        edgecolor="black",
        linewidth=1.5,
    )

    ax.set_ylabel("Average Performance Score", fontsize=12)
    ax.set_title("Model Performance Summary", fontsize=14, fontweight="bold")
    ax.set_ylim([0, 1])
    ax.grid(axis="y", alpha=0.3)

    # Add value labels
    for bar, val in zip(bars, df["Avg_Score"]):
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2.0,
            height + 0.02,
            f"{val:.3f}",
            ha="center",
            va="bottom",
            fontsize=11,
            fontweight="bold",
        )

    # Add recommendation text
    ax.text(
        0.5,
        -0.15,
        f"🏆 Best Model: {best_model} (Average Score: {best_score:.3f})",
        transform=ax.transAxes,
        ha="center",
        va="center",
        fontsize=13,
        fontweight="bold",
        bbox=dict(boxstyle="round", facecolor="#F18F01", alpha=0.3),
    )

    plt.tight_layout()
    plt.show()

    # Print detailed results
    print(f"\n{'=' * 60}")
    print(f" RECOMMENDATION: {best_model} is the best performing model")
    print(f"{'=' * 60}")
    print(f"\n📊 Key Metrics for {best_model}:")
    print(f"   • Accuracy:  {df.loc[best_model, 'Accuracy']:.4f}")
    print(f"   • Precision: {df.loc[best_model, 'Precision']:.4f}")
    print(f"   • Recall:    {df.loc[best_model, 'Recall']:.4f}")
    print(f"   • F1-Score:  {df.loc[best_model, 'F1-Score']:.4f}")
    print(f"   • ROC-AUC:   {df.loc[best_model, 'ROC-AUC']:.4f}")
    print(f"\n{'=' * 60}")


select_best_model(all_metrics)

#### Summary

In [ ]:
print("\n" + "=" * 70)
print(" COMPARATIVE EVALUATION OF DEEP LEARNING MODELS")
print(" FOR BRAIN STROKE DETECTION")
print("=" * 70)

print("\n📁 DATASET SUMMARY:")
print(f"   • Training images:   {train_generator.samples}")
print(f"   • Validation images: {validation_generator.samples}")
print(f"   • Testing images:    {test_generator.samples}")
print(f"   • Classes:           {list(train_generator.class_indices.keys())}")

print("\n🤖 MODELS EVALUATED:")
print("   • VGG16 - Classic architecture with 16 layers")
print("   • DenseNet201 - Dense connectivity pattern")
print("   • ResNet50 - Residual connections")
print("   • EfficientNetB0 - Efficient scaling")

print("\n📊 PERFORMANCE SUMMARY:")
df_summary = pd.DataFrame(all_metrics)
df_summary = df_summary.round(4)
print(df_summary.to_string(index=False))

print("\n🔍 EXPLAINABILITY METHODS IMPLEMENTED:")
print("   • Grad-CAM - Visual attention maps")
print("   • LIME - Local explanations")
print("   • SHAP - Game-theoretic feature attribution")

print("\n" + "=" * 70)
print(" ANALYSIS COMPLETE")
print("=" * 70)